# El Geometrisi Biyometrik Tanıma — Derin Öğrenme Modelleri

Bu notebook **MBA-Net**, **ABD-Net** ve **RGA-Net** mimarilerini 11K Hands veri setinde çalıştırır.

**Çalıştırma öncesi:**
1. Runtime → Change runtime type → **T4 GPU** seç
2. Hücreleri sırayla çalıştır

In [ ]:
# Hücre 1 — Bağımlılık Kurulumu
!pip install kagglehub -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA kullanılabilir: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Hücre 2 — Kaggle Token ile Dataset İndirme

import os, shutil

# ↓↓↓ SADECE BURAYA TOKEN'INI YAPISTIR ↓↓↓
KAGGLE_TOKEN = "KGAT_buraya_kendi_tokenini_yapistir"
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

import kagglehub
print("Dataset indiriliyor (1-2 dk sürebilir)...")
cache_path = kagglehub.dataset_download("shyambhu/hands-and-palm-images-dataset")
print(f"İndirildi: {cache_path}")

DATA_DIR   = "/content/data/11k_hands"
IMAGE_ROOT = os.path.join(DATA_DIR, "Hands", "Hands")
os.makedirs(IMAGE_ROOT, exist_ok=True)

copied = 0
for root, _, fnames in os.walk(cache_path):
    for fname in fnames:
        src = os.path.join(root, fname)
        dst = os.path.join(
            IMAGE_ROOT if fname.lower().endswith((".jpg", ".jpeg", ".png")) else DATA_DIR,
            fname
        )
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1

print(f"{copied} dosya kopyalandı.")
print(f"Görüntü sayısı: {len(os.listdir(IMAGE_ROOT))}")

In [ ]:
# Hücre 3 — Import'lar ve Sabitler
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import euclidean

HANDINFO_CSV = os.path.join(DATA_DIR, "HandInfo.csv")
OUTPUT_DIR   = "/content/output"
PLOTS_DIR    = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE       = 224
BATCH_SIZE     = 64
EPOCHS         = 30
LR_INIT        = 1e-4
LR_STEP        = 10
LR_GAMMA       = 0.5
EMBED_DIM      = 512
TRIPLET_MARGIN = 0.3
TRIPLET_WEIGHT = 0.5
MIN_SAMPLES    = 10
TEST_SPLIT     = 0.30
RANDOM_STATE   = 42
FREEZE_EPOCHS  = 5

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print(f"Device: {DEVICE}")

In [ ]:
# Hücre 4 — HandBiometricDataset

TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class HandBiometricDataset(Dataset):
    def __init__(self, handinfo_csv, image_root, aspect_filter=None,
                 transform=None, min_samples=MIN_SAMPLES):
        df = pd.read_csv(handinfo_csv)

        # Sütun adlarını normalize et
        df.columns = [c.strip() for c in df.columns]
        img_col = next(c for c in df.columns if "image" in c.lower() or "file" in c.lower())
        id_col  = next(c for c in df.columns if "id" in c.lower() and "image" not in c.lower())
        asp_col = next(c for c in df.columns if "aspect" in c.lower())

        if aspect_filter:
            df = df[df[asp_col].str.lower().str.contains(aspect_filter.lower())].copy()

        counts = df[id_col].value_counts()
        valid  = counts[counts >= min_samples].index
        df     = df[df[id_col].isin(valid)].copy()

        le = LabelEncoder()
        df["label"] = le.fit_transform(df[id_col].astype(str))

        self.samples      = list(zip(
            df[img_col].apply(lambda f: os.path.join(image_root, f)),
            df["label"].tolist()
        ))
        self.transform    = transform
        self.label_encoder = le
        self._labels      = df["label"].to_numpy()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_labels(self):
        return self._labels

    @property
    def num_classes(self):
        return len(self.label_encoder.classes_)


def make_loaders(aspect_filter="palmar"):
    full_ds = HandBiometricDataset(HANDINFO_CSV, IMAGE_ROOT, aspect_filter=aspect_filter)
    labels  = full_ds.get_labels()
    n       = len(full_ds)

    tr_idx, te_idx = train_test_split(
        np.arange(n), test_size=TEST_SPLIT, stratify=labels, random_state=RANDOM_STATE
    )

    train_ds = HandBiometricDataset(HANDINFO_CSV, IMAGE_ROOT, aspect_filter=aspect_filter,
                                    transform=TRAIN_TRANSFORM)
    test_ds  = HandBiometricDataset(HANDINFO_CSV, IMAGE_ROOT, aspect_filter=aspect_filter,
                                    transform=TEST_TRANSFORM)

    train_loader = DataLoader(Subset(train_ds, tr_idx), batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(Subset(test_ds,  te_idx), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f"Aspect: {aspect_filter or 'tum'} | "
          f"Sınıf: {full_ds.num_classes} | "
          f"Eğitim: {len(tr_idx)} | Test: {len(te_idx)}")
    return train_loader, test_loader, full_ds.num_classes


# Test
tr_loader, te_loader, n_cls = make_loaders("palmar")

In [ ]:
# Hücre 5 — Paylaşılan Bloklar

class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1, 1)
        return x * s


class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.mlp(self.avg_pool(x).view(b, c))
        mx  = self.mlp(self.max_pool(x).view(b, c))
        att = torch.sigmoid(avg + mx).view(b, c, 1, 1)
        return x * att


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max( dim=1, keepdim=True).values
        att = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * att


class TripletLoss(nn.Module):
    def __init__(self, margin=TRIPLET_MARGIN):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        # Kare uzaklık matrisi: (B, B)
        diff = embeddings.unsqueeze(0) - embeddings.unsqueeze(1)
        dist = (diff ** 2).sum(dim=2)

        # Aynı / farklı sınıf maskeleri
        labels = labels.unsqueeze(1)
        same   = (labels == labels.T).float()
        diff_m = 1 - same
        # Diyagonali sıfırla (kendisiyle karşılaştırma)
        eye    = torch.eye(dist.size(0), device=dist.device)
        same   = same - eye

        # Hardest positive: en uzak aynı sınıf örneği
        pos_dist = (dist * same).max(dim=1).values

        # Semi-hard negative: anchor-positive'den büyük, en küçük negatif
        big_val = dist.max().item() + 1
        neg_dist_masked = dist + big_val * (same + eye)
        # Sadece d_neg > d_pos olanlar (semi-hard)
        mask = (dist > pos_dist.unsqueeze(1)).float() * diff_m
        neg_dist = torch.where(
            mask.bool(),
            dist,
            torch.full_like(dist, big_val)
        ).min(dim=1).values

        loss = F.relu(pos_dist - neg_dist + self.margin).mean()
        return loss


print("Paylaşılan bloklar tanımlandı.")

In [ ]:
# Hücre 6 — MBA-Net (Multi-Branch Attention Network)

class MBANet(nn.Module):
    def __init__(self, num_classes, embed_dim=EMBED_DIM, num_stripes=4):
        super().__init__()
        backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        # layer4 çıktısına kadar al (avgpool ve fc hariç)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.se       = SEBlock(2048)
        self.num_stripes = num_stripes

        # Global branch
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.global_bn   = nn.BatchNorm1d(2048)
        self.global_drop = nn.Dropout(0.5)
        self.global_fc   = nn.Linear(2048, embed_dim)

        # Local branch (şeritler)
        stripe_dim = embed_dim // num_stripes
        self.stripe_pools = nn.ModuleList([
            nn.AdaptiveAvgPool2d((1, 1)) for _ in range(num_stripes)
        ])
        self.stripe_fcs = nn.ModuleList([
            nn.Linear(2048, stripe_dim) for _ in range(num_stripes)
        ])
        self.local_bn  = nn.BatchNorm1d(embed_dim)
        self.local_drop = nn.Dropout(0.3)

        # Classifier (global + local birleşimi)
        self.classifier = nn.Linear(2 * embed_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)     # (B, 2048, 7, 7)
        feat = self.se(feat)

        # Global
        g = self.global_pool(feat).flatten(1)   # (B, 2048)
        g = self.global_drop(self.global_bn(g))
        g = self.global_fc(g)                   # (B, embed_dim)

        # Local: H boyutunda şeritlere böl
        H = feat.size(2)
        stripe_h = H // self.num_stripes
        stripes = []
        for i in range(self.num_stripes):
            start = i * stripe_h
            end   = (i + 1) * stripe_h if i < self.num_stripes - 1 else H
            s = self.stripe_pools[i](feat[:, :, start:end, :]).flatten(1)  # (B, 2048)
            s = F.relu(self.stripe_fcs[i](s))
            stripes.append(s)
        l = torch.cat(stripes, dim=1)           # (B, embed_dim)
        l = self.local_drop(self.local_bn(l))

        embedding = torch.cat([g, l], dim=1)    # (B, 2*embed_dim)
        embedding = F.normalize(embedding, p=2, dim=1)
        logits    = self.classifier(embedding)
        return logits, embedding


print("MBA-Net tanımlandı.")

In [ ]:
# Hücre 7 — ABD-Net (Attentive but Diverse Network)

class ABDNet(nn.Module):
    def __init__(self, num_classes, embed_dim=EMBED_DIM, lambda_div=1e-3):
        super().__init__()
        self.lambda_div = lambda_div
        backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])

        # Attention branch
        self.ch_att  = ChannelAttention(2048)
        self.sp_att  = SpatialAttention()
        self.att_pool = nn.AdaptiveAvgPool2d(1)
        self.att_bn  = nn.BatchNorm1d(2048)
        self.att_fc  = nn.Linear(2048, embed_dim)

        # Backbone branch (dikkat yok — çeşitlilik için)
        self.base_pool = nn.AdaptiveAvgPool2d(1)
        self.base_bn   = nn.BatchNorm1d(2048)
        self.base_fc   = nn.Linear(2048, embed_dim)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)    # (B, 2048, 7, 7)

        # Attention branch
        a = self.ch_att(feat)
        a = self.sp_att(a)
        a = self.att_pool(a).flatten(1)
        a = self.att_bn(a)
        a = self.att_fc(a)         # (B, embed_dim)

        # Backbone branch
        b = self.base_pool(feat).flatten(1)
        b = self.base_bn(b)
        b = self.base_fc(b)        # (B, embed_dim)

        # Diversity loss: attention ve backbone branch ağırlıklarının ortogonalliği
        W_att  = self.att_fc.weight   # (embed_dim, 2048)
        W_base = self.base_fc.weight  # (embed_dim, 2048)
        div_loss = (torch.mm(W_att, W_base.T) -
                    torch.eye(W_att.size(0), device=W_att.device)).pow(2).sum()

        embedding = F.normalize(a, p=2, dim=1)
        logits    = self.classifier(embedding)
        return logits, embedding, div_loss


print("ABD-Net tanımlandı.")

In [ ]:
# Hücre 8 — RGA-Net (Relation-Guided Attention Network)

class RGAModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # İlişki-rehberli özelliği orijinal özellikle birleştiren conv
        self.merge = nn.Conv2d(2 * in_channels, in_channels, kernel_size=1, bias=False)
        self.bn    = nn.BatchNorm2d(in_channels)
        self.gate  = nn.Sigmoid()

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        F_flat = x.view(B, C, N)                           # (B, C, N)

        # Uzamsal öz-ilişki matrisi
        R = torch.bmm(F_flat.permute(0, 2, 1), F_flat)    # (B, N, N)
        R = F.softmax(R, dim=2)                             # satır normaliz.

        # İlişki-ağırlıklı özellik toplama
        F_rel = torch.bmm(F_flat, R.permute(0, 2, 1))     # (B, C, N)
        F_rel = F_rel.view(B, C, H, W)

        # Birleştir ve sigmoid gating uygula
        merged = self.merge(torch.cat([x, F_rel], dim=1))
        merged = self.bn(merged)
        scale  = self.gate(merged)
        return x * scale


class RGANet(nn.Module):
    def __init__(self, num_classes, embed_dim=EMBED_DIM):
        super().__init__()
        bb = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.layer0 = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1 = bb.layer1
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3
        self.layer4 = bb.layer4

        # RGA modülleri layer3 (1024-ch, 14×14) ve layer4 (2048-ch, 7×7) sonrasına
        # layer2 atlanır: 28×28=784 pozisyon → 784×784 matris bellek taşırır
        self.rga3 = RGAModule(1024)
        self.rga4 = RGAModule(2048)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.bn   = nn.BatchNorm1d(2048)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(2048, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.rga3(x)            # (B, 1024, 14, 14)
        x = self.layer4(x)
        x = self.rga4(x)            # (B, 2048, 7, 7)

        x = self.pool(x).flatten(1) # (B, 2048)
        x = self.drop(self.bn(x))
        embedding = F.normalize(self.fc(x), p=2, dim=1)
        logits    = self.classifier(embedding)
        return logits, embedding


print("RGA-Net tanımlandı.")

In [ ]:
# Hücre 9 — Eğitim Döngüsü

def train_one_epoch(model, loader, optimizer, ce_loss, triplet_loss, model_name):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        if model_name == "ABD-Net":
            logits, embs, div_loss = model(imgs)
            loss = (ce_loss(logits, labels)
                    + TRIPLET_WEIGHT * triplet_loss(embs, labels)
                    + model.lambda_div * div_loss)
        else:
            logits, embs = model(imgs)
            loss = (ce_loss(logits, labels)
                    + TRIPLET_WEIGHT * triplet_loss(embs, labels))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, model_name):
    model.eval()
    all_embs, all_labels, all_preds = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        if model_name == "ABD-Net":
            logits, embs, _ = model(imgs)
        else:
            logits, embs = model(imgs)
        all_embs.append(embs.cpu().numpy())
        all_labels.append(labels.numpy())
        all_preds.append(logits.argmax(1).cpu().numpy())

    embs   = np.concatenate(all_embs)
    labels = np.concatenate(all_labels)
    preds  = np.concatenate(all_preds)
    acc    = (preds == labels).mean()
    return acc, embs, labels


def train_model(model, train_loader, test_loader, model_name):
    optimizer    = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=5e-4)
    scheduler    = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)
    ce_loss      = nn.CrossEntropyLoss()
    triplet_loss = TripletLoss(TRIPLET_MARGIN)
    history      = {"train_loss": [], "train_acc": [], "test_acc": []}

    # İlk FREEZE_EPOCHS epoch backbone dondurulur
    def set_backbone_grad(requires_grad):
        attr = "backbone" if hasattr(model, "backbone") else "layer0"
        for name, param in model.named_parameters():
            if name.startswith(("backbone", "layer0", "layer1", "layer2",
                                 "layer3", "layer4", "rga3", "rga4", "se",
                                 "ch_att", "sp_att")):
                param.requires_grad = requires_grad

    set_backbone_grad(False)

    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            set_backbone_grad(True)
            print(f"  → Epoch {epoch+1}: backbone açıldı")

        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, ce_loss, triplet_loss, model_name)
        te_acc, _, _ = evaluate(model, test_loader, model_name)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)

        print(f"  Epoch {epoch+1:02d}/{EPOCHS} | "
              f"loss={tr_loss:.4f} | "
              f"train={tr_acc*100:.2f}% | "
              f"test={te_acc*100:.2f}%")

    _, te_embs, te_labels = evaluate(model, test_loader, model_name)
    _, tr_embs, tr_labels = evaluate(model, train_loader, model_name)
    return model, history, tr_embs, tr_labels, te_embs, te_labels


print("Eğitim fonksiyonları tanımlandı.")

In [ ]:
# Hücre 10 — Metrik Hesaplama

def compute_cmc(tr_embs, tr_labels, te_embs, te_labels, max_rank=10):
    from sklearn.metrics.pairwise import euclidean_distances
    dist = euclidean_distances(te_embs, tr_embs)   # (n_probe, n_gallery)
    cmc  = np.zeros(max_rank)
    for i, lbl in enumerate(te_labels):
        sorted_lbl = tr_labels[np.argsort(dist[i])]
        for rank in range(max_rank):
            if sorted_lbl[rank] == lbl:
                cmc[rank:] += 1
                break
    return cmc / len(te_labels)


def compute_eer(tr_embs, tr_labels, te_embs, te_labels, n_thresholds=500):
    classes   = np.unique(tr_labels)
    centroids = {c: tr_embs[tr_labels == c].mean(axis=0) for c in classes}

    genuine_scores, impostor_scores = [], []
    rng = np.random.default_rng(RANDOM_STATE)

    for i in range(len(te_embs)):
        probe, true_lbl = te_embs[i], te_labels[i]
        if true_lbl not in centroids:
            continue
        genuine_scores.append(-euclidean(probe, centroids[true_lbl]))
        others = [c for c in classes if c != true_lbl]
        for c in rng.choice(others, size=min(10, len(others)), replace=False):
            impostor_scores.append(-euclidean(probe, centroids[c]))

    gen = np.array(genuine_scores)
    imp = np.array(impostor_scores)
    all_sc     = np.concatenate([gen, imp])
    thresholds = np.linspace(all_sc.min(), all_sc.max(), n_thresholds)

    far = np.array([np.mean(imp >= t) for t in thresholds])
    frr = np.array([np.mean(gen <  t) for t in thresholds])
    idx = np.argmin(np.abs(far - frr))
    eer = (far[idx] + frr[idx]) / 2.0

    return thresholds, far, frr, eer, thresholds[idx], gen, imp


print("Metrik fonksiyonları tanımlandı.")

In [ ]:
# Hücre 11 — Görselleştirme Fonksiyonları

def plot_training_history(history, tag):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history["train_loss"], color="steelblue", linewidth=2)
    ax1.set_title(f"Eğitim Kaybı — {tag}"); ax1.set_xlabel("Epoch"); ax1.grid(alpha=0.4)
    ax2.plot(history["train_acc"], label="Eğitim", color="steelblue", linewidth=2)
    ax2.plot(history["test_acc"],  label="Test",   color="coral",    linewidth=2)
    ax2.set_title(f"Doğruluk — {tag}"); ax2.set_xlabel("Epoch")
    ax2.legend(); ax2.grid(alpha=0.4)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"dl_training_{tag}.png")
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f"  [grafik] {os.path.basename(path)}")


def plot_far_frr(thresholds, far, frr, eer, eer_thr, tag):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(thresholds, far, color="red",  linewidth=2, label="FAR (Yanlış Kabul)")
    ax.plot(thresholds, frr, color="blue", linewidth=2, label="FRR (Yanlış Red)")
    ax.axvline(eer_thr, color="green", linestyle="--", linewidth=1.5,
               label=f"EER = {eer*100:.2f}%")
    ax.set_xlabel("Eşik"); ax.set_ylabel("Hata Oranı")
    ax.set_title(f"FAR & FRR — {tag}"); ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"dl_far_frr_{tag}.png")
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f"  [grafik] {os.path.basename(path)}")


def plot_cmc_curves(cmc_dict, tag):
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = ["steelblue", "coral", "seagreen"]
    for (name, cmc), color in zip(cmc_dict.items(), colors):
        ax.plot(range(1, len(cmc)+1), cmc * 100, marker="o",
                linewidth=2, color=color, label=name)
    ax.set_xlabel("Rank"); ax.set_ylabel("Tanıma Oranı (%)")
    ax.set_title(f"CMC Eğrisi — {tag}"); ax.legend(); ax.grid(alpha=0.4)
    ax.set_xticks(range(1, 11))
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"dl_cmc_{tag}.png")
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f"  [grafik] {os.path.basename(path)}")


def plot_dl_comparison(dl_results, tag):
    df = pd.DataFrame(dl_results)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    colors = ["steelblue", "coral", "seagreen"]

    ax1.bar(df["Model"], df["Rank1"] * 100, color=colors[:len(df)])
    ax1.set_ylabel("Rank-1 (%)"); ax1.set_title(f"Rank-1 Doğruluğu — {tag}")
    ax1.set_ylim(0, 100); ax1.grid(axis="y", alpha=0.4)
    for i, v in enumerate(df["Rank1"] * 100):
        ax1.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)

    ax2.bar(df["Model"], df["EER"] * 100, color=colors[:len(df)])
    ax2.set_ylabel("EER (%)"); ax2.set_title(f"EER — {tag}")
    ax2.set_ylim(0, 60); ax2.grid(axis="y", alpha=0.4)
    for i, v in enumerate(df["EER"] * 100):
        ax2.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)

    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f"dl_comparison_{tag}.png")
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f"  [grafik] {os.path.basename(path)}")


print("Görselleştirme fonksiyonları tanımlandı.")

In [ ]:
# Hücre 12 — Deney Çalıştırıcı

def run_experiment(model_class, model_name, aspect_filter="palmar"):
    tag = f"{model_name}_{aspect_filter or 'tum'}"
    print(f"\n{'='*60}")
    print(f"  DENEY: {tag}")
    print(f"{'='*60}")

    train_loader, test_loader, num_classes = make_loaders(aspect_filter)

    model = model_class(num_classes=num_classes).to(DEVICE)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parametre sayısı: {params:,}")

    model, history, tr_embs, tr_labels, te_embs, te_labels = train_model(
        model, train_loader, test_loader, model_name
    )

    # Metrikler
    cmc = compute_cmc(tr_embs, tr_labels, te_embs, te_labels)
    thresholds, far, frr, eer, eer_thr, gen_sc, imp_sc = compute_eer(
        tr_embs, tr_labels, te_embs, te_labels
    )
    final_acc = history["test_acc"][-1]

    print(f"\n  Sonuçlar:")
    print(f"    Rank-1  : {cmc[0]*100:.2f}%")
    print(f"    Accuracy: {final_acc*100:.2f}%")
    print(f"    EER     : {eer*100:.2f}%")

    # Grafikler
    plot_training_history(history, tag)
    plot_far_frr(thresholds, far, frr, eer, eer_thr, tag)

    # Modeli kaydet
    model_path = os.path.join(OUTPUT_DIR, f"{tag}_model.pt")
    torch.save(model.state_dict(), model_path)
    print(f"  Model kaydedildi: {model_path}")

    return {
        "Model":    model_name,
        "Aspect":   aspect_filter or "tum",
        "Rank1":    round(cmc[0], 4),
        "Accuracy": round(final_acc, 4),
        "EER":      round(eer, 4),
        "Params":   params,
    }, cmc


print("Deney çalıştırıcı tanımlandı.")

In [ ]:
# Hücre 13 — Tüm Modelleri Çalıştır (Palmar)
# Her model ~5-10 dk sürer (T4 GPU ile)

MODELS = [
    (MBANet,  "MBA-Net"),
    (ABDNet,  "ABD-Net"),
    (RGANet,  "RGA-Net"),
]
ASPECT = "palmar"

all_results = []
cmc_curves  = {}

for model_class, model_name in MODELS:
    result, cmc = run_experiment(model_class, model_name, aspect_filter=ASPECT)
    all_results.append(result)
    cmc_curves[model_name] = cmc

# CMC karşılaştırma grafiği
plot_cmc_curves(cmc_curves, ASPECT)

# Bar grafik karşılaştırması
plot_dl_comparison(all_results, ASPECT)

# Sonuç tablosu
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "dl_results_summary.csv"), index=False)
print("\n" + "="*60)
print("SONUÇ TABLOSU")
print("="*60)
display(results_df)

In [ ]:
# Hücre 14 (İsteğe Bağlı) — Dorsal Aspect İçin Tekrar Çalıştır

dorsal_results = []
dorsal_cmc     = {}

for model_class, model_name in MODELS:
    result, cmc = run_experiment(model_class, model_name, aspect_filter="dorsal")
    dorsal_results.append(result)
    dorsal_cmc[model_name] = cmc

plot_cmc_curves(dorsal_cmc, "dorsal")
plot_dl_comparison(dorsal_results, "dorsal")
display(pd.DataFrame(dorsal_results))